# numpy, and why the shape is the error

MichAl Academy, lesson 1.3.

Run each cell with **Shift+Enter**. numpy is already installed in Colab and
Kaggle, so there is nothing to set up.

Work the broadcasting section by hand before you run it. Guessing and checking
teaches the rule far better than reading it does.

## 1. An array is not a list

In [ ]:
import numpy as np

lengths = np.array([12, 47, 8, 31])

print(lengths)
print("dtype:", lengths.dtype)
print("shape:", lengths.shape)

Every element has the same type, and the array knows it. That is the trade: a
list will hold anything and grow, an array is fixed and uniform, and the
uniformity is what lets the work happen in compiled code instead of a Python
loop.

In [ ]:
print(lengths * 2)
print(lengths + 100)
print(lengths > 20)

No loop. The operation applied to the whole array at once.

`lengths > 20` gave back an array of `True` and `False` with the same shape.
That is a mask, and you can index with it.

In [ ]:
mask = lengths > 20

print("mask: ", mask)
print("kept: ", lengths[mask])
print("count:", mask.sum())      # True counts as 1

`mask.sum()` counting the `True` values is a trick you will use constantly:
how many rows were flagged, how many predictions were correct.

## 2. Shape

A shape is one number per axis. Read `(3, 4)` as three of something, each
holding four.

In [ ]:
counts = np.array([[1, 2, 3, 4],
                   [5, 6, 7, 8],
                   [9, 0, 1, 2]])

print(counts)
print("shape:", counts.shape)
print("ndim: ", counts.ndim)
print("size: ", counts.size)

In [ ]:
print("first row:      ", counts[0])
print("first column:   ", counts[:, 0])
print("row 1, column 2:", counts[1, 2])
print("last two columns:")
print(counts[:, -2:])

The comma separates axes. `counts[1, 2]` is row 1, column 2, both counted from
0. A colon on its own means "all of this axis".

`reshape` hands back the same numbers arranged differently. It does not change
any values.

In [ ]:
flat = np.arange(12)
print(flat)
print(flat.reshape(3, 4))
print(flat.reshape(3, 4).shape)

## 3. Broadcasting

The rule, from the NumPy user guide: start at the rightmost axis and work left.
Two axes fit if they are equal, or if one of them is 1. An axis an array does
not have counts as 1.

Work these out on paper first. Write down what you expect, then run the cell.

In [ ]:
pairs = [((3, 4), (4,)),
         ((3, 4), (3,)),
         ((3, 1), (1, 4)),
         ((2, 3, 4), (4,)),
         ((2, 3, 4), (3, 1, 4))]

for a, b in pairs:
    try:
        out = np.broadcast_shapes(a, b)
        print(f"{str(a):>12} + {str(b):<10} -> {out}")
    except ValueError as e:
        print(f"{str(a):>12} + {str(b):<10} -> FAILS: {e}")

`np.broadcast_shapes` answers the question without building any arrays, which
makes it a good way to check yourself. It words the failure differently from
the message you get when you actually add two arrays, so expect to meet both.

How many did you get right? The second one is the one that catches people.

## 4. The classic mistake

You have a table of three rows and four columns, and you want to adjust it.

In [ ]:
counts = np.array([[1, 2, 3, 4],
                   [5, 6, 7, 8],
                   [9, 0, 1, 2]])

per_column = np.array([10, 20, 30, 40])     # shape (4,)

print(counts + per_column)

That worked. Alignment starts from the right, `4` met `4`, and the same four
numbers were added to every row.

Now try one number per **row** instead.

In [ ]:
per_row = np.array([100, 200, 300])         # shape (3,)

try:
    print(counts + per_row)
except ValueError as e:
    print("ValueError:", e)

Your intent is obvious to you and invisible to numpy. It aligned from the right
as always, compared your `3` against the table's `4`, and stopped.

The fix is to give the array the axis it is missing, so there is no ambiguity
about which way it should stretch.

In [ ]:
print("per_row shape:         ", per_row.shape)
print("reshaped:              ", per_row.reshape(3, 1).shape)
print()
print(counts + per_row.reshape(3, 1))

`(3, 1)` against `(3, 4)`: the 3s match, the 1 stretches across the four
columns, and each row gets its own number. The values never changed, only how
many axes the array claims to have.

`per_row[:, np.newaxis]` does the same thing and you will see both.

## 5. Your turn

This function is supposed to scale every row so that the row sums to 1. It runs
without any error at all, and it is wrong.

Run it, look at the output, then work out what the shapes are doing.

In [ ]:
def normalise_rows(table):
    totals = table.sum(axis=1)     # TODO: what shape is this?
    return table / totals


t = np.array([[1., 1., 2., 4.],
              [2., 2., 2., 2.],
              [0., 1., 1., 2.],
              [3., 1., 0., 0.]])

out = normalise_rows(t)

print("row sums:", out.sum(axis=1))
print("every row sums to 1?", np.allclose(out.sum(axis=1), 1))

Print `table.shape` and `totals.shape` inside the function if you are stuck.
The table is square, which is exactly why nothing complained.

Change one line so the last print says `True`.

<details>
<summary>Answer</summary>

`totals` has shape `(4,)`, so it aligned with the **columns**, not the rows.
Give it the missing axis:

```python
def normalise_rows(table):
    totals = table.sum(axis=1).reshape(-1, 1)   # shape (4, 1)
    return table / totals
```

Or ask for it up front, which is the version you will see most often:

```python
def normalise_rows(table):
    totals = table.sum(axis=1, keepdims=True)   # shape (4, 1)
    return table / totals
```

</details>

## What you now have

- An array is fixed in size and uniform in type, and operations apply to all of it at once
- A mask of `True` and `False` selects rows, and `.sum()` counts them
- `.shape` is the first thing to print when something is wrong
- Broadcasting aligns from the right, and a missing axis counts as 1
- A square table can hide a shape bug, because the wrong axis still fits

Next is lesson 1.4, pandas, which is this with column names on it.